#### 다음 실습 코드는 학습 목적으로만 사용 바랍니다. 문의 : audit@korea.ac.kr 임성열 Ph.D.

In [ ]:

# python -m venv llm
# pip install -r requirements-llm.txt
# 이미 설치했다면 아래 셀은 실행하지 않습니다.
# ------------------------------------------------------------

# %pip install torch numpy

In [ ]:
# PyTorch 라이브러리 불러오기
# --------------------------------------------------
# torch        : PyTorch 기본 라이브러리
# torch.nn     : 신경망(Layer) 구현 모듈
# torch.optim  : 최적화(Optimizer) 알고리즘
# numpy        : 수치 계산 라이브러리
 
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np


# 학습 데이터 준비
# --------------------------------------------------
# 문자열 데이터를 문자(Character) 단위로 학습하기 위한
# 데이터셋을 생성합니다.

text = "hello world machine learning is fun "

# 등장하는 문자 목록 추출
chars = sorted(list(set(text)))

# 문자 -> 숫자(ID) 변환 딕셔너리, 딥러닝 모델은 문자를 직접 이해하지 못하므로 숫자로 변환
char_to_idx = {c: i for i, c in enumerate(chars)}

# 숫자(ID) -> 문자 변환 딕셔너리, 모델이 예측을 끝낸 후, 다시 문자로 변환하여 처리
idx_to_char = {i: c for i, c in enumerate(chars)}


# 모델 하이퍼파라미터 설정
# --------------------------------------------------
# 모델 구조 및 학습에 사용할 주요 설정값입니다.

input_size = len(chars)      # 문자 종류 개수(Vocabulary Size)
hidden_size = 50             # LSTM 은닉층 크기, LSTM의 기억 공간이 50칸,LSTM이 문장의 정보를 저장하는 내부 메모리의 크기 
num_layers = 2               # LSTM 층 개수, LSTM을 한 번 거칠지, 여러 번 거칠지를 결정
seq_length = 10              # 입력 시퀀스 길이, 한 번에 입력할 문자 개수
learning_rate = 0.01         # 학습률, 0.01만큼 조금씩 가중치를 수정하며 학습, 값이 크면 빠르게 학습하지만 최적의 값을 지나칠 수 있고 값이 작으면 천천히 안정적으로 학습하지만 시간이 오래 걸림


# 학습 데이터 생성
# --------------------------------------------------
# 입력 시퀀스와 다음 문자를 예측하기 위한 정답 시퀀스를
# 생성합니다.

input_data = []
target_data = []

for i in range(0, len(text) - seq_length):

    # 입력 문자열
    input_seq = text[i:i+seq_length]

    # 한 글자씩 앞으로 이동한 정답 문자열
    target_seq = text[i+1:i+seq_length+1]

    # 문자를 숫자로 변환하여 저장
    input_data.append([char_to_idx[c] for c in input_seq])
    target_data.append([char_to_idx[c] for c in target_seq])

# PyTorch Tensor로 변환, LSTM은 리스트를 계산할 수 없으므로 PyTorch 모델이 사용할 수 있는 자료형(Tensor)으로 변환
input_data = torch.LongTensor(input_data)
target_data = torch.LongTensor(target_data)


# 문자 생성(Character Generation) LSTM 모델 정의
# --------------------------------------------------
# Embedding -> LSTM -> Linear 구조를 이용하여
# 다음 문자를 예측하는 모델입니다.

class CharLSTM(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers):

        super(CharLSTM, self).__init__()

        # 문자 ID -> 임베딩 벡터
        self.embedding = nn.Embedding(input_size, hidden_size)

        # LSTM 계층
        self.lstm = nn.LSTM(
            hidden_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        # 출력층
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x, hidden):

        # 문자 임베딩
        x = self.embedding(x)

        # LSTM 수행
        out, hidden = self.lstm(x, hidden)

        # 다음 문자 예측
        out = self.fc(out)

        return out, hidden


# 모델 생성
model = CharLSTM(input_size, hidden_size, num_layers)

# 손실 함수 및 최적화 알고리즘 설정
# --------------------------------------------------
# CrossEntropyLoss : 다중 클래스 분류 손실 함수
# Adam             : 가중치 업데이트 알고리즘

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

# 모델 학습
# --------------------------------------------------
# 입력 문장을 이용하여 다음 문자를 예측하도록
# LSTM 모델을 반복 학습합니다.

for epoch in range(200):

    hidden = None

    # 모델 실행
    output, hidden = model(input_data, hidden)

    # 손실 계산
    loss = criterion(
        output.view(-1, input_size),
        target_data.view(-1)
    )

    # 이전 Gradient 초기화
    optimizer.zero_grad()

    # 역전파
    loss.backward()

    # 가중치 업데이트
    optimizer.step()

    # 20회마다 Loss 출력
    if epoch % 20 == 0:
        print(f"Epoch [{epoch}/200] Loss: {loss.item():.4f}")


# 문장 생성 함수
# --------------------------------------------------
# 학습된 모델을 이용하여 시작 문자열 이후의
# 문자를 순차적으로 생성합니다.

def generate_text(start_str, length):

    # 추론(Inference) 모드
    model.eval()

    # 시작 문자열을 숫자로 변환
    chars_input = torch.LongTensor(
        [[char_to_idx[c] for c in start_str]]
    )

    hidden = None

    result = start_str

    # 지정한 길이만큼 반복 생성
    for _ in range(length):

        output, hidden = model(chars_input, hidden)

        # 마지막 시점의 출력 사용
        last_char_logits = output[:, -1, :]

        # 가장 높은 확률의 문자 선택
        last_char_idx = torch.argmax(
            last_char_logits,
            dim=1
        ).item()

        # 결과 문자열에 추가
        result += idx_to_char[last_char_idx]

        # 생성된 문자를 다음 입력으로 사용
        chars_input = torch.LongTensor([[last_char_idx]])

    return result

# 결과 확인
# --------------------------------------------------
# "hello"를 시작으로 새로운 문장을 생성합니다.

print("\nGenerated Text:")
print(generate_text("hello", 50))

Epoch [0/200] Loss: 2.8296
Epoch [20/200] Loss: 0.2920
Epoch [40/200] Loss: 0.0970
Epoch [60/200] Loss: 0.0853
Epoch [80/200] Loss: 0.0830
Epoch [100/200] Loss: 0.0820
Epoch [120/200] Loss: 0.0814
Epoch [140/200] Loss: 0.0809
Epoch [160/200] Loss: 0.0806
Epoch [180/200] Loss: 0.0803

Generated Text:
hello world machine learning is fun is fun is fun is fu
